# Vessel Detection Inference

This notebook provides simplified inference functionality for PyNAS trained vessel detection models.

In [ ]:
"""Simplified inference script for PyNAS trained vessel detection model."""

import os
import sys
sys.path.append('..')
from pathlib import Path
from typing import Optional, Dict

import torch
import numpy as np
import matplotlib.pyplot as plt

from datasets.RawVessels.loader import RawVesselsDataModule

In [ ]:
class VesselInference:
    """Simplified vessel detection inference class."""
    
    def __init__(self, model_path: str, device: Optional[str] = None) -> None:
        """
        Initialize the inference class.
        
        Args:
            model_path (str): Path to the TorchScript model file
            device (Optional[str]): Device to run inference on. If None, auto-detect
        """
        self.model_path = Path(model_path)
        assert self.model_path.exists(), f'Model file not found: {model_path}'
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') if device is None else torch.device(device)
        print(f'Using device: {self.device}')
        
        self.model = torch.jit.load(self.model_path, map_location=self.device)
        self.model.eval()
        print(f'Model loaded from {self.model_path}')
    
    def predict(self, input_tensor: torch.Tensor, threshold: float = 0.5) -> np.ndarray:
        """
        Perform inference on input tensor.
        
        Args:
            input_tensor (torch.Tensor): Input tensor (B, C, H, W)
            threshold (float): Threshold for binary classification
            
        Returns:
            np.ndarray: Binary prediction masks (B, H, W)
        """
        input_tensor = input_tensor.to(self.device)
        
        with torch.no_grad():
            predictions = self.model(input_tensor)
        
        # Handle different output formats
        if predictions.shape[1] == 2:
            vessel_probs = predictions[:, 1, :, :]
        else:
            vessel_probs = predictions.squeeze(1)
        
        # Apply sigmoid if needed
        if vessel_probs.min() < 0 or vessel_probs.max() > 1:
            vessel_probs = torch.sigmoid(vessel_probs)
        
        binary_masks = (vessel_probs.cpu().numpy() > threshold).astype(np.uint8)
        return binary_masks
    
    def calculate_metrics(self, predictions: np.ndarray, targets: np.ndarray) -> Dict[str, float]:
        """
        Calculate basic evaluation metrics.
        
        Args:
            predictions (np.ndarray): Binary prediction masks
            targets (np.ndarray): Ground truth masks
            
        Returns:
            Dict[str, float]: Dictionary of metrics
        """
        pred_flat = predictions.flatten()
        target_flat = targets.flatten()
        
        tp = np.sum((pred_flat == 1) & (target_flat == 1))
        fp = np.sum((pred_flat == 1) & (target_flat == 0))
        fn = np.sum((pred_flat == 0) & (target_flat == 1))
        tn = np.sum((pred_flat == 0) & (target_flat == 0))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
        
        return {'precision': precision, 'recall': recall, 'f1_score': f1_score, 'iou': iou}
    
    def visualize(self, image: np.ndarray, prediction: np.ndarray, target: Optional[np.ndarray] = None) -> None:
        """
        Visualize inference results.
        
        Args:
            image (np.ndarray): Input image (H, W)
            prediction (np.ndarray): Prediction mask (H, W)
            target (Optional[np.ndarray]): Ground truth mask (H, W)
        """
        cols = 3 if target is not None else 2
        fig, axes = plt.subplots(1, cols, figsize=(5 * cols, 5))
        
        if cols == 2:
            axes = [axes[0], axes[1]]
        
        axes[0].imshow(image, cmap='gray')
        axes[0].set_title('Input Image')
        axes[0].axis('off')
        
        axes[1].imshow(prediction, cmap='gray')
        axes[1].set_title('Prediction')
        axes[1].axis('off')
        
        if target is not None:
            axes[2].imshow(target, cmap='gray')
            axes[2].set_title('Ground Truth')
            axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
    
    def visualize2(self, image: np.ndarray, prediction: np.ndarray, target: np.ndarray, 
                   metrics: Optional[Dict[str, float]] = None, pred_alpha: float = 0.3, 
                   gt_alpha: float = 0.3, idx: int = 0) -> None:
        """
        Improved visualization with GT on left and prediction on right.
        
        Args:
            image (np.ndarray): Input image (H, W)
            prediction (np.ndarray): Prediction mask (H, W)
            target (np.ndarray): Ground truth mask (H, W)
            metrics (Optional[Dict[str, float]]): Metrics to display
            pred_alpha (float): Alpha value for prediction overlay
            gt_alpha (float): Alpha value for ground truth overlay
        """
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # Left plot: Ground Truth overlay
        axes[0].imshow(image, cmap='gray')
        
        gt_overlay = np.zeros((*target.shape, 4))
        gt_overlay[target == 1] = [0, 1, 0, gt_alpha]  # Green for ground truth
        axes[0].imshow(gt_overlay)
        axes[0].set_title('Ground Truth (Green)', fontsize=14, fontweight='bold')
        axes[0].axis('off')
        
        # Right plot: Prediction overlay
        axes[1].imshow(image, cmap='gray')
        
        pred_overlay = np.zeros((*prediction.shape, 4))
        pred_overlay[prediction == 1] = [1, 0, 0, pred_alpha]  # Red for predictions
        axes[1].imshow(pred_overlay)
        axes[1].set_title('Prediction (Red)', fontsize=14, fontweight='bold')
        axes[1].axis('off')
        
        # Add metrics text if provided
        if metrics is not None:
            metrics_text = '\n'.join([f'{k.replace("_", " ").title()}: {v:.3f}' for k, v in metrics.items()])
            axes[1].text(0.02, 0.98, metrics_text, transform=axes[1].transAxes, 
                        fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', 
                        facecolor='white', alpha=0.8))
        
        plt.tight_layout()
        basedir = '/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/Outputs'
        plt.savefig(f'{basedir}/inference_result_{idx}.png', dpi=300)
        plt.show()


In [ ]:
# Configuration
model_path = '/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/Results/models_traced/generation_9/model_and_architecture_26.pt'
data_dir = '/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/data/TASI/DataSAR_real_refined'
batch_size = 1
threshold = 0.5

# Initialize inference
inferencer = VesselInference(model_path)
# Setup data module
data_module = RawVesselsDataModule(root_dir=data_dir, batch_size=batch_size)
data_module.setup()

print(f'Data module setup complete:')
print(f'  Input shape: {data_module.input_shape}')
print(f'  Number of classes: {data_module.num_classes}')
print('=' * 50)
print(f'  Train dataset size: {len(data_module.train_dataset):,}')
print(f'  Validation dataset size: {len(data_module.val_dataset):,}')
print(f'  Test dataset size: {len(data_module.test_dataset):,}')
print('=' * 50)
print(f'  Batch size: {batch_size}')
print(f'  Data directory: {data_dir}')

In [ ]:

# Get a test sample and run inference
test_loader = data_module.test_dataloader()
# Get multiple test samples for comparison
test_samples = []
for i, (batch_images, batch_targets) in enumerate(test_loader):
    test_samples.append((batch_images, batch_targets))
    if i >= 240:  # Get 40 samples total
        break

# Select sample at index 2
for idx in range(len(test_samples)):
    images, targets = test_samples[idx]

    # Run inference
    predictions = inferencer.predict(images, threshold=threshold)

    # Extract data for visualization
    image = images[0, 0].numpy()  # First image, first channel
    prediction = predictions[0]   # First prediction
    target = targets[0, 1].numpy() if targets.shape[1] == 2 else targets[0].squeeze().numpy()

    # Calculate and display metrics
    metrics = inferencer.calculate_metrics(predictions, target.reshape(1, *target.shape))

    # Visualize results
    inferencer.visualize2(image, prediction, target, pred_alpha=0.6, gt_alpha=0.6, metrics=metrics, idx=idx)
    print(f'Metrics: {metrics}')


# # Get a test sample and run inference
# test_loader = data_module.test_dataloader()
# # Get multiple test samples for comparison
# test_samples = []
# for i, (batch_images, batch_targets) in enumerate(test_loader):
#     test_samples.append((batch_images, batch_targets))
#     if i >= 240:  # Get 40 samples total
#         break

# # Select sample at index 2
# idx = 4
# images, targets = test_samples[idx]

# # Run inference
# predictions = inferencer.predict(images, threshold=threshold)

# # Extract data for visualization
# image = images[0, 0].numpy()  # First image, first channel
# prediction = predictions[0]   # First prediction
# target = targets[0, 1].numpy() if targets.shape[1] == 2 else targets[0].squeeze().numpy()

# # Calculate and display metrics
# metrics = inferencer.calculate_metrics(predictions, target.reshape(1, *target.shape))

# # Visualize results
# inferencer.visualize2(image, prediction, target, pred_alpha=0.6, gt_alpha=0.6, metrics=metrics, idx=idx)
# print(f'Metrics: {metrics}')